# UN Comtrade 一括取得 — HS 6桁・輸出

| 項目 | 設定 |
|---|---|
| 品目 | HS **1〜24類の6桁コード**（1,176件）※`CHAPTERS` で変更可 |
| 分類 | HS (as reported) = `clCode="HS"` |
| 報告国 | 全報告国 |
| 相手国 | **World（全世界合計）のみ** ※`PARTNER` で変更可 |
| フロー | **輸出のみ**（`X`） |
| 期間 | **2012〜2025年**（14年）※`YEARS` で変更可 |

**年 × 類**の単位（336ブロック）で取得し、`data/ex_ch01-24_2012-2025/` に
`ch01_2012.csv.gz` のような gzip CSV で保存する。中断しても再開できる。

各ファイルは `cmdCode` 昇順（`hs_codes.csv` の id 順）→ 報告国 → 相手国 で並ぶ。

### 見積もり

- 件数 : 約 120万件（2025年実測 約89,500件 × 14年）
- 時間 : 約 40〜60分
- 容量 : 30MB前後（gzip後）

## 1. セットアップ

In [1]:
import os
import time
from pathlib import Path

import pandas as pd
from dotenv import load_dotenv

import comtradeapicall as c

load_dotenv(Path.home() / ".comtrade_env")
KEY = (os.environ.get("COMTRADE_KEY") or "").strip()
if not KEY:
    raise RuntimeError("COMTRADE_KEY が未設定。~/.comtrade_env を確認すること。")

pd.set_option("display.max_columns", 30)
pd.set_option("display.width", 220)
print(f"キー検出（{len(KEY)} 文字）")

キー検出（32 文字）


## 2. 設定

`CHAPTERS` が対象の類、`OUTDIR` はそこから自動生成される。

In [2]:
YEARS    = [str(y) for y in range(2012, 2026)]     # 2012〜2025年（14年分）
FLOW     = "X"                                    # 輸出のみ
CHAPTERS = [f"{i:02d}" for i in range(1, 25)]     # 1〜24類
PARTNER  = None                                   # None=全相手国 / "0"=World（全世界合計）のみ

# 対象を変えるときはここを書き換える
#   YEARS    = ["2025"]                              単年
#   YEARS    = [str(y) for y in range(2000, 2026)]   2000〜2025年
#   CHAPTERS = ["01", "02"]                          一部の類だけ

# 出力先は CHAPTERS / YEARS / PARTNER から自動で決まる（対象を変えても混ざらない）。
# PARTNER を含めるのは、World のみ版と全相手国版で行数が1桁違うため。同じフォルダに
# 混ざると再開判定が古い条件のファイルを「取得済み」と誤認してスキップしてしまう。
_ctag  = f"ch{CHAPTERS[0]}" if len(CHAPTERS) == 1 else f"ch{CHAPTERS[0]}-{CHAPTERS[-1]}"
_ytag  = YEARS[0] if len(YEARS) == 1 else f"{YEARS[0]}-{YEARS[-1]}"
_ptag  = "world" if PARTNER == "0" else "allp"
OUTDIR = Path("data") / f"ex_{_ctag}_{_ytag}_{_ptag}"

# 保存する18列（API 側の実際の列名。qty / cifvalue / fobvalue は小文字始まり）
COLS = [
    "refYear", "period", "reporterCode", "reporterDesc", "flowDesc",
    "partnerCode", "partnerDesc", "classificationCode", "cmdCode", "cmdDesc",
    "qtyUnitAbbr", "qty", "altQty", "netWgt", "grossWgt",
    "cifvalue", "fobvalue", "primaryValue",
]

# 分割の判断に使う2つの上限（どちらも実測値）
CAP            = 100_000   # 1リクエストの最大レコード数。超過分は黙って捨てられる
CODE_STR_MAX   = 1_200     # cmdCode 文字列の上限。URL全体が2000文字を超えると弾かれる
PAUSE          = 0.3       # リクエスト間の待機（秒）

OUTDIR.mkdir(parents=True, exist_ok=True)
print(f"対象年 : {YEARS[0]}〜{YEARS[-1]}（{len(YEARS)}年）")
print(f"対象類 : 第{CHAPTERS[0]}〜{CHAPTERS[-1]}類（{len(CHAPTERS)}類）")
print(f"取得単位: {len(YEARS) * len(CHAPTERS)} ブロック（年 × 類）")
print(f"相手国 : {'World のみ' if PARTNER == '0' else '全相手国'}")
print("出力先 :", OUTDIR.resolve())

対象年 : 2012〜2025（14年）
対象類 : 第01〜24類（24類）
取得単位: 336 ブロック（年 × 類）
相手国 : 全相手国
出力先 : /Users/nakasukadaiki/Projects/comtrade/data/ex_ch01-24_2012-2025_allp


## 3. 対象コードの抽出

`getReference("cmd:HS")` から `aggrLevel == 6`（6桁）かつ `CHAPTERS` に該当するものだけを取り出す。

In [3]:
HS_CSV = Path("data") / "hs_codes.csv"
if HS_CSV.exists():
    HS = pd.read_csv(HS_CSV, dtype={"id": str, "parent": str})
else:
    HS = c.getReference("cmd:HS")
    Path("data").mkdir(exist_ok=True)
    HS.to_csv(HS_CSV, index=False, encoding="utf-8-sig")

six = HS[HS["aggrLevel"] == 6].copy()
six["ch"] = six["id"].astype(str).str[:2]

CODES = {ch: sorted(six.loc[six["ch"] == ch, "id"].astype(str)) for ch in CHAPTERS}
total_codes = sum(len(v) for v in CODES.values())

print(f"対象6桁コード: {total_codes:,} 件")
print({ch: len(v) for ch, v in CODES.items()})

対象6桁コード: 1,176 件
{'01': 44, '02': 81, '03': 282, '04': 38, '05': 17, '06': 22, '07': 79, '08': 84, '09': 56, '10': 33, '11': 35, '12': 63, '13': 12, '14': 14, '15': 70, '16': 45, '17': 19, '18': 11, '19': 20, '20': 63, '21': 17, '22': 26, '23': 28, '24': 17}


## 4. 取得ロジック

対応している制約は3つ。

1. **URL 2000文字** — `cmdCode` が長すぎると `Request URL exceeds maximum allowed length`
   で弾かれる。第03類（282コード＝1,973文字）が実際に失敗した。
2. **1リクエスト100,000件** — `maxRecords` に何を指定しても超過分は**警告なしに捨てられる**。
   打ち切られた場合は**ちょうど100,000件**返るので、それ未満なら欠落なしと判断できる。
3. **呼び出し回数のクォータ** — 使い切ると 403 が返る（復活まで十数時間）。
   ライブラリは例外を出さず `None` を返すだけなので、自前で検出して明示的に停止する。

コード列を再帰的に二分し、**文字数と件数の両方が収まるまで**分割してから取得する。

In [4]:
import contextlib
import io
import re


class QuotaExceeded(RuntimeError):
    """API の呼び出し回数クォータを使い切った。"""


def _args(codes_str, year, **over):
    a = dict(
        typeCode="C", freqCode="A", clCode="HS", period=year,
        reporterCode=None, cmdCode=codes_str, flowCode=FLOW,
        partnerCode=PARTNER, partner2Code="0", customsCode="C00", motCode="0",
    )
    a.update(over)
    return a


def _call(fn, *args, **kwargs):
    """API を呼び、ライブラリが標準出力に印字するエラーを拾う。

    comtradeapicall は失敗時に例外ではなく「メッセージを print して None を返す」ため、
    標準出力を捕まえないと原因が分からない。403（クォータ切れ）だけは専用の例外にする。
    """
    buf = io.StringIO()
    with contextlib.redirect_stdout(buf):
        res = fn(*args, **kwargs)
    msg = buf.getvalue().strip()
    if msg:
        if "403" in msg or "call volume quota" in msg:
            m = re.search(r"replenished in ([0-9:]+)", msg)
            raise QuotaExceeded(f"呼び出し回数のクォータ切れ。復活まで {m.group(1) if m else '不明'}")
        print(f"    [API] {msg[:160]}")
    return res


def count_of(codes_str, year, **over):
    """該当件数を返す。取れなければ None。（検証用。通常の取得では使わない）"""
    r = _call(c.getCountFinalData, KEY, **_args(codes_str, year, **over))
    if r is None or getattr(r, "empty", True):
        return None
    return int(r["count"].iloc[0])


def fetch_raw(codes_str, year, **over):
    """1リクエスト分を取得する。空なら None。"""
    df = _call(c.getFinalData, KEY, maxRecords=CAP, includeDesc=True,
               **_args(codes_str, year, **over))
    if df is None or getattr(df, "empty", True):
        return None
    return df


def fetch_codes(codes, year, label="", depth=0):
    """コード列を必要なだけ分割して取得し、DataFrame のリストを返す。

    件数照会は行わない。打ち切りはちょうど CAP 件で起きるため、
    取得結果が CAP 未満なら欠落なしと判断でき、リクエスト数が半分で済む。
    """
    codes_str = ",".join(codes)
    indent = "  " * (depth + 1)

    # ① URL長で分割
    if len(codes_str) > CODE_STR_MAX and len(codes) > 1:
        mid = len(codes) // 2
        return (fetch_codes(codes[:mid], year, label, depth + 1)
                + fetch_codes(codes[mid:], year, label, depth + 1))

    df = fetch_raw(codes_str, year)
    time.sleep(PAUSE)
    got = 0 if df is None else len(df)

    # ② 上限ちょうど = 打ち切りの可能性 → 分割して取り直す
    if got >= CAP:
        if len(codes) > 1:
            mid = len(codes) // 2
            print(f"{indent}上限 {CAP:,} 件に到達 → {len(codes)}コードを二分割")
            return (fetch_codes(codes[:mid], year, label, depth + 1)
                    + fetch_codes(codes[mid:], year, label, depth + 1))
        # 単一コードで超過 → 報告国で分ける
        print(f"{indent}単一コード {codes[0]} が上限到達 → 報告国で分割")
        reps = sorted(c.getReference("reporter")["reporterCode"].astype(str).unique())
        out = []
        for i in range(0, len(reps), 40):
            grp = ",".join(reps[i:i + 40])
            d = fetch_raw(codes_str, year, reporterCode=grp)
            time.sleep(PAUSE)
            if d is not None:
                if len(d) >= CAP:
                    raise RuntimeError(f"{label}: {codes[0]} は報告国分割でも上限に達した")
                out.append(d)
        return out

    if got:
        print(f"{indent}{len(codes):>3}コード → {got:>7,} 件")
    return [df] if got else []

## 5. 実行

**年 × 類**の単位で取得して即保存する（`ch01_2012.csv.gz` の粒度）。
保存済みブロックは自動でスキップされるので、**このセルを再実行すれば必ず続きから走る**。

API のクォータが切れた場合もトレースバックにはならず、進捗を表示して停止する。
復活後にこのセルをもう一度実行すればよい。

In [10]:
t_start = time.time()
summary = []
blocks = [(y, ch) for y in YEARS for ch in CHAPTERS]
todo = [(y, ch) for y, ch in blocks if not (OUTDIR / f"ch{ch}_{y}.csv.gz").exists()]

print(f"全 {len(blocks)} ブロック中 {len(blocks)-len(todo)} 件は取得済み。残り {len(todo)} 件を処理する。\n")
stopped = None

for k, (year, ch) in enumerate(todo, 1):
    out_path = OUTDIR / f"ch{ch}_{year}.csv.gz"
    codes = CODES[ch]
    print(f"[{k}/{len(todo)}] {year} 第{ch}類 — {len(codes)}コード")
    t0 = time.time()

    try:
        frames = fetch_codes(codes, year, label=f"{year}/ch{ch}")
    except QuotaExceeded as e:
        stopped = e
        break

    if not frames:
        print("  データなし")
        summary.append((year, ch, 0, 0.0))
        continue

    df_ch = pd.concat(frames, ignore_index=True)

    missing = [col for col in COLS if col not in df_ch.columns]
    if missing:
        raise RuntimeError(f"列が存在しない: {missing}")
    df_ch = df_ch[COLS]

    # hs_codes.csv の id 順（cmdCode 昇順）に並べ替える。
    # API の返却順は報告国コード順で、しかも保証されていないため自前で固定する。
    # 6桁ゼロ埋めの文字列なので、辞書順＝数値順になる。
    df_ch = (df_ch
             .astype({"cmdCode": str})
             .sort_values(["cmdCode", "reporterCode", "partnerCode"], kind="stable")
             .reset_index(drop=True))

    df_ch.to_csv(out_path, index=False, encoding="utf-8-sig", compression="gzip")
    print(f"  → {len(df_ch):,} 行 / {time.time()-t0:.0f}秒 / {out_path.name}")
    summary.append((year, ch, len(df_ch), time.time() - t0))

done = len(list(OUTDIR.glob("ch*_*.csv.gz")))
print(f"\n{'='*60}")
if stopped:
    print(f"■ 中断: {stopped}")
    print("  → クォータ復活後にこのセルを再実行すれば続きから走る。")
else:
    print("■ 完了")
print(f"進捗: {done}/{len(blocks)} ブロック（残り {len(blocks)-done}）")
print(f"経過: {time.time()-t_start:.0f} 秒")
if summary:
    s = pd.DataFrame(summary, columns=["year", "chapter", "rows", "sec"])
    print(f"今回取得: {s['rows'].sum():,} 行")
    display(s.groupby("year", as_index=False)["rows"].sum())

全 336 ブロック中 218 件は取得済み。残り 118 件を処理する。

[1/118] 2021 第03類 — 282コード
    141コード →  37,718 件
    141コード →  52,386 件
  → 90,104 行 / 375秒 / ch03_2021.csv.gz
[2/118] 2021 第04類 — 38コード
   38コード →  46,934 件
  → 46,934 行 / 188秒 / ch04_2021.csv.gz
[3/118] 2021 第05類 — 17コード
   17コード →  10,733 件
  → 10,733 行 / 49秒 / ch05_2021.csv.gz
[4/118] 2021 第06類 — 22コード
   22コード →  16,751 件
  → 16,751 行 / 55秒 / ch06_2021.csv.gz
[5/118] 2021 第07類 — 79コード
   79コード →  73,535 件
  → 73,535 行 / 244秒 / ch07_2021.csv.gz
[6/118] 2021 第08類 — 84コード
   84コード →  78,002 件
  → 78,002 行 / 438秒 / ch08_2021.csv.gz
[7/118] 2021 第09類 — 56コード
   56コード →  64,579 件
  → 64,579 行 / 200秒 / ch09_2021.csv.gz
[8/118] 2021 第10類 — 33コード
   33コード →  22,420 件
  → 22,420 行 / 27秒 / ch10_2021.csv.gz
[9/118] 2021 第11類 — 35コード
   35コード →  33,264 件
  → 33,264 行 / 40秒 / ch11_2021.csv.gz
[10/118] 2021 第12類 — 63コード
   63コード →  44,011 件
  → 44,011 行 / 53秒 / ch12_2021.csv.gz
[11/118] 2021 第13類 — 12コード
   12コード →  12,127 件
  → 12,127 行 / 16秒 / ch13_2021.

,year,rows
0,2021,936817
1,2022,991563
2,2023,998708
3,2024,962240
4,2025,830576


## 6. 結合と確認

In [11]:
# ファイル名が ch{類}_{年}.csv.gz なので、sorted() だと類→年の順に並ぶ。
# 年→類→cmdCode の順に揃えたいので、読み込み後にまとめてソートする。
files = sorted(OUTDIR.glob("ch*_*.csv.gz"))
print(f"{len(files)} ファイル")

df = pd.concat(
    [pd.read_csv(f, dtype={"cmdCode": str, "classificationCode": str}) for f in files],
    ignore_index=True)

# 年ごとに、各年の中は取得時と同じ並び（cmdCode 昇順 → 報告国 → 相手国）
df = (df.sort_values(["period", "cmdCode", "reporterCode", "partnerCode"], kind="stable")
        .reset_index(drop=True))

print(f"\n合計 {len(df):,} 行 x {df.shape[1]} 列")
print(f"列: {list(df.columns)}")

336 ファイル

合計 12,873,073 行 x 18 列
列: ['refYear', 'period', 'reporterCode', 'reporterDesc', 'flowDesc', 'partnerCode', 'partnerDesc', 'classificationCode', 'cmdCode', 'cmdDesc', 'qtyUnitAbbr', 'qty', 'altQty', 'netWgt', 'grossWgt', 'cifvalue', 'fobvalue', 'primaryValue']


In [5]:
# 検証
print("品目コードの桁数:", sorted(df["cmdCode"].astype(str).str.len().unique()), " ← [6] であること")
print("類の範囲       :", sorted(df["cmdCode"].astype(str).str[:2].unique()))
print("フロー         :", sorted(df["flowDesc"].astype(str).unique()), " ← Export のみ")
print("年             :", sorted(df["refYear"].astype(str).unique()))
print("相手国         :", sorted(df["partnerDesc"].astype(str).unique()), " ← World のみ")
print("報告国数       :", df["reporterCode"].nunique())
print("品目数         :", df["cmdCode"].nunique(), f"/ 対象 {total_codes}")
print()
print("年ごとの行数:")
print(df.groupby("refYear").size().to_string())

dup = df.duplicated(subset=["period", "reporterCode", "partnerCode", "cmdCode", "flowDesc"]).sum()
print("\n重複行:", dup, " ← 0 であること")

NameError: name 'df' is not defined

In [6]:
# comtrade_start.ipynb と同じ列・同じ順序で表示する
# （保存ファイルは COLS の18列のまま。ここは画面表示用の抜粋）
VIEW = ["period", "reporterCode", "reporterDesc", "flowDesc",
        "partnerCode", "partnerDesc", "cmdCode", "cmdDesc",
        "qty", "qtyUnitAbbr", "netWgt", "primaryValue"]

df[[col for col in VIEW if col in df.columns]].head(20)

NameError: name 'df' is not defined

In [14]:
# 1ファイルにまとめる場合
ALL = OUTDIR / f"ex_{_ctag}_{_ytag}_{_ptag}_all.csv.gz"
df.to_csv(ALL, index=False, encoding="utf-8-sig", compression="gzip")
print("保存:", ALL.resolve(), f"({ALL.stat().st_size/1e6:.1f} MB / {len(df):,} 行)")

保存: /Users/nakasukadaiki/Projects/comtrade/data/ex_ch01-24_2012-2025_allp/ex_ch01-24_2012-2025_allp_all.csv.gz (276.2 MB / 12,873,073 行)


## 7. 年ごとに1ファイルへまとめる

取得結果は `ch{類}_{年}.csv.gz`（類 × 年）の粒度で保存されている。
これを**年ごと**に結合し、`data/by_year_{相手国設定}/trade_{年}.csv.gz` を作る。

各ブロック内は `cmdCode` 昇順だが、24類を単純に連結すると類の境界で並びが切れるため、
結合後に `cmdCode` → 報告国 → 相手国 で並べ直す。

保存済みの年はスキップするので、途中で止めても再実行すれば続きから走る。

In [9]:
# 分割元。取得直後は OUTDIR のままでよい。
# フォルダを移動・改名した場合はここを書き換える（例: Path("data/trade_data")）。
SRC_DIR = Path("data/trade_data")
if not SRC_DIR.exists():
    raise FileNotFoundError(f"取得結果が見つからない: {SRC_DIR}  ← SRC_DIR を実際の場所に書き換えること")

BYYEAR = Path("data") / f"by_year_{_ptag}"
BYYEAR.mkdir(parents=True, exist_ok=True)
print("分割元:", SRC_DIR.resolve())
print("出力先:", BYYEAR.resolve())
print()

rows_total = 0
t_all = time.time()

for year in YEARS:
    out_path = BYYEAR / f"trade_{year}.csv.gz"
    if out_path.exists():
        print(f"[skip] {year} — 保存済み")
        continue

    t0 = time.time()
    paths = [SRC_DIR / f"ch{ch}_{year}.csv.gz" for ch in CHAPTERS]
    lack = [p.name for p in paths if not p.exists()]
    if lack:
        raise FileNotFoundError(f"{year}: ブロックが欠けている {lack}")

    # dtype 指定は必須。省くと 0201 が 201 になり先頭ゼロが落ちる。
    df_y = pd.concat(
        [pd.read_csv(p, dtype={"cmdCode": str, "classificationCode": str}) for p in paths],
        ignore_index=True)

    df_y = (df_y.sort_values(["cmdCode", "reporterCode", "partnerCode"], kind="stable")
                .reset_index(drop=True))

    df_y.to_csv(out_path, index=False, encoding="utf-8-sig", compression="gzip")
    rows_total += len(df_y)
    print(f"{year}: {len(df_y):>9,} 行 / {out_path.stat().st_size/1e6:6.1f} MB / {time.time()-t0:5.1f}秒")

print(f"\n完了: {time.time()-t_all:.0f} 秒")
print(f"出力: {len(list(BYYEAR.glob('trade_*.csv.gz')))} ファイル / 今回 {rows_total:,} 行")

分割元: /Users/nakasukadaiki/Projects/comtrade/data/trade_data
出力先: /Users/nakasukadaiki/Projects/comtrade/data/by_year_allp

[skip] 2012 — 保存済み
2013:   844,982 行 /   17.3 MB /  15.1秒
2014:   854,522 行 /   17.5 MB /  14.7秒
2015:   888,341 行 /   18.4 MB /  15.3秒
2016:   895,330 行 /   18.8 MB /  15.8秒
2017:   929,121 行 /   20.5 MB /  18.5秒
2018:   942,869 行 /   20.8 MB /  17.9秒
2019:   966,335 行 /   21.2 MB /  20.3秒
2020:   962,846 行 /   20.9 MB /  19.8秒
2021:   994,665 行 /   21.7 MB /  21.3秒
2022:   991,563 行 /   21.7 MB /  20.4秒
2023:   998,708 行 /   21.8 MB /  19.4秒
2024:   962,240 行 /   21.0 MB /  19.5秒
2025:   830,576 行 /   18.2 MB /  16.5秒

完了: 235 秒
出力: 14 ファイル / 今回 12,062,098 行


In [8]:
# 検証 — 年ごとファイルの行数が、元ブロックの合計と一致するか
import gzip

print(f"{'年':<6}{'年ファイル':>12}{'元ブロック計':>14}  一致")
ok = True
for year in YEARS:
    f = BYYEAR / f"trade_{year}.csv.gz"
    if not f.exists():
        continue
    with gzip.open(f, "rt", encoding="utf-8-sig") as fh:
        n_year = sum(1 for _ in fh) - 1
    n_src = 0
    for ch in CHAPTERS:
        with gzip.open(SRC_DIR / f"ch{ch}_{year}.csv.gz", "rt", encoding="utf-8-sig") as fh:
            n_src += sum(1 for _ in fh) - 1
    match = n_year == n_src
    ok &= match
    print(f"{year:<6}{n_year:>12,}{n_src:>14,}  {'OK' if match else '★不一致'}")

print("\n" + ("すべて一致" if ok else "★不一致あり — 再作成すること"))

年            年ファイル        元ブロック計  一致


FileNotFoundError: [Errno 2] No such file or directory: 'data/ex_ch01-24_2012-2025_allp/ch01_2012.csv.gz'

## メモ

- **`cifvalue` は輸出データではほぼ空**（CIF は輸入側の評価額）。輸出額は `fobvalue`
  または `primaryValue` を使う。
- `partnerDesc == "World"` は全世界合計。個別相手国と足すと**二重計上**になる。
- `partner2Code="0"` / `customsCode="C00"` / `motCode="0"` は合計行だけを取るための指定。
  外すと輸送モード別・通関手続別・原産国別の内訳が混ざって数倍に膨れる。
- 対象を変えるには `YEARS` / `CHAPTERS` を書き換える（出力先も自動で分かれる）。
- 途中で止まったら同じセルを再実行すれば、保存済みの類を飛ばして続きから走る。
  特定のブロックをやり直したいときは `OUTDIR` 内の `ch{類}_{年}.csv.gz` を消してから再実行する。